# 11 — Feature Engineering Datos Físicos
Mismo esquema que NB 07 (red): eliminar ruido, crear features nuevas, normalizar.
Resultado: Delta `features_fisicos/` listo para modelo y para concatenar con features de red.

In [0]:
from pyspark.sql import functions as F
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Rutas
DELTA_NORMAL_PATH  = "/Volumes/workspace/default/phisical_measures/delta_normal/"
DELTA_ATTACK_PATH  = "/Volumes/workspace/default/phisical_measures/delta_attack/"
DELTA_FE_FISICOS   = "/Volumes/workspace/default/phisical_measures/features_fisicos/"

# Cargar y unir con label binario — mismo esquema que en red
df_normal = spark.read.format("delta").load(DELTA_NORMAL_PATH)
df_attack  = spark.read.format("delta").load(DELTA_ATTACK_PATH)

df = df_normal.withColumn("label", F.lit(0)).union(
    df_attack.withColumn("label",
        F.when(F.col("Normal_Attack") == "Attack", 1).otherwise(0)
    )
).drop("Normal_Attack", "Timestamp")

print(f"Total registros : {df.count():,}")
print(f"Columnas        : {len(df.columns)}")

## 1 — Eliminar sensor sin señal

In [0]:
# FIT601 tuvo correlación con el label prácticamente 0 en el EDA
# No aporta información discriminativa → se elimina igual que hicimos
# con las features de ruido en los datos de red (NB 07)
df = df.drop("FIT601")

# Sensores que quedan con señal real confirmada por EDA
SIGNAL_COLS = ["LIT401", "DPIT301", "LIT101", "FIT201", "FIT101", "LIT301"]

print(f"Sensores eliminados : ['FIT601']")
print(f"Sensores con señal  : {SIGNAL_COLS}")

## 2 — Normalización MinMaxScaler

In [0]:
# Los 6 sensores tienen escalas muy distintas:
#   LIT101, LIT301, LIT401 → niveles en mm  (rango ~0-1200)
#   FIT101, FIT201         → caudales m3/h  (rango ~0-3)
#   DPIT301                → presión kPa    (rango ~0-50)
#
# Si no normalizamos, al concatenar con features de red (que sí están
# en rangos similares) los sensores de nivel dominarían por magnitud.
# MinMaxScaler lleva todos al rango [0, 1] manteniendo la distribución.

# Traer a pandas para aplicar sklearn
pdf = df.select(SIGNAL_COLS + ["label", "timestamp_dt"]).toPandas()
pdf[SIGNAL_COLS] = pdf[SIGNAL_COLS].astype(float)
pdf["label"]     = pdf["label"].astype(int)

# Ajustar scaler solo sobre los datos de entrenamiento sería lo ideal,
# pero como el objetivo es normalizar el rango físico del sensor
# (no aprender de los datos de train), lo ajustamos sobre todo el dataset
scaler = MinMaxScaler()
pdf[SIGNAL_COLS] = scaler.fit_transform(pdf[SIGNAL_COLS])

print("Rangos tras MinMaxScaler:")
print(pdf[SIGNAL_COLS].describe().loc[["min", "max"]].to_string())

## 3 — Crear features nuevas

In [0]:
# ── FLAGS BINARIOS ──────────────────────────────────────────────────────────
# Basados en los umbrales que los histogramas del EDA mostraron claramente.
# Son equivalentes a is_short_duration en los de red: capturan zonas del
# espacio de valores donde una clase domina claramente sobre la otra.

# LIT401 < 0.5 (en escala normalizada) → zona de ataque
# En el histograma: Normal en 800-1000 mm, Ataque en ~250 mm
# El umbral 600 mm ≈ 0.5 en [0,1]
pdf["lit401_low"] = (pdf["LIT401"] < 0.5).astype(int)

# FIT201 < 0.1 → caudal cortado (zona de ataque)
# En el histograma: Normal en ~2.5 m3/h, Ataque masivamente en ~0
pdf["fit201_zero"] = (pdf["FIT201"] < 0.1).astype(int)

# FIT101 < 0.1 → mismo patrón que FIT201
# Caudal de entrada etapa 1 cortado durante ataque
pdf["fit101_zero"] = (pdf["FIT101"] < 0.1).astype(int)

# LIT101 > 0.7 → nivel tanque 1 en zona alta (ataque lleva nivel al límite)
# En el histograma: Normal disperso 200-800 mm, Ataque concentrado en ~850
pdf["lit101_high"] = (pdf["LIT101"] > 0.7).astype(int)

# LIT301 > 0.85 → nivel tanque 3 en zona alta
# En el histograma: Normal en 750-1050 mm, Ataque en 1000-1050
pdf["lit301_high"] = (pdf["LIT301"] > 0.85).astype(int)

# DPIT301 < 0.1 → presión diferencial colapsada
# En el histograma: Normal tiene picos en ~1 y ~20 kPa, Ataque en ~0
pdf["dpit301_low"] = (pdf["DPIT301"] < 0.1).astype(int)

flag_cols = ["lit401_low", "fit201_zero", "fit101_zero",
             "lit101_high", "lit301_high", "dpit301_low"]

print("Distribución de flags por clase:")
print(
    pd.DataFrame({
        col: [
            pdf[pdf["label"]==0][col].mean().round(3),
            pdf[pdf["label"]==1][col].mean().round(3)
        ]
        for col in flag_cols
    }, index=["Normal", "Ataque"])
    .to_string()
)

In [0]:
# ── RATIOS ──────────────────────────────────────────────────────────────────
# Combinaciones físicamente significativas entre sensores relacionados.
# Son equivalentes a duration_ratio y packets_per_ms en los de red:
# capturan relaciones entre variables que por separado no bastan.

# Relación nivel tanque 4 / caudal entrada etapa 2
# Si el nivel baja y el caudal también → anomalía física clara
# Suma 0.01 para evitar división por cero cuando FIT201 ≈ 0
pdf["lit401_fit201_ratio"] = pdf["LIT401"] / (pdf["FIT201"] + 0.01)

# Relación entre caudales de etapas consecutivas (1 y 2)
# En operación normal deberían estar equilibrados
# Un desequilibrio grande indica manipulación de alguno de los dos
pdf["fit101_fit201_ratio"] = pdf["FIT101"] / (pdf["FIT201"] + 0.01)

# Relación entre niveles de los tanques extremos (etapa 1 y etapa 4)
# Captura si el desequilibrio del sistema es global o local
pdf["lit101_lit401_ratio"] = pdf["LIT101"] / (pdf["LIT401"] + 0.01)

ratio_cols = ["lit401_fit201_ratio", "fit101_fit201_ratio", "lit101_lit401_ratio"]

print("Medias de ratios por clase:")
print(
    pd.DataFrame({
        col: [
            pdf[pdf["label"]==0][col].mean().round(4),
            pdf[pdf["label"]==1][col].mean().round(4)
        ]
        for col in ratio_cols
    }, index=["Normal", "Ataque"])
    .to_string()
)

## 4 — Verificar correlación de nuevas features

In [0]:
# Comprobar que las features creadas tienen señal real con el label
# Mismo análisis que hicimos en NB 07 para los de red
nuevas_cols = flag_cols + ratio_cols
all_feature_cols = SIGNAL_COLS + nuevas_cols

corr_nuevas = (
    pdf[all_feature_cols + ["label"]]
    .corr()["label"]
    .drop("label")
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
corr_nuevas.columns = ["feature", "corr_abs"]

print("Correlación con label — features originales + nuevas:")
print(corr_nuevas.to_string(index=False))

# Visualización
fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ["#E8453C" if v > 0.1 else "#4C8BF5"
              for v in corr_nuevas["corr_abs"]]
ax.barh(corr_nuevas["feature"][::-1],
        corr_nuevas["corr_abs"][::-1],
        color=bar_colors[::-1], edgecolor="white")
ax.axvline(0.1, color="gray", linestyle="--", lw=0.8, label="umbral 0.1")
ax.set_title("Correlacion con label — post Feature Engineering",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Correlacion absoluta")
ax.legend()
plt.tight_layout()
plt.show()

## 5 — Distribución de nuevas features por clase

In [0]:
pdf_n = pdf[pdf["label"] == 0]
pdf_a = pdf[pdf["label"] == 1]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes_flat = axes.flatten()

for i, col in enumerate(nuevas_cols):
    ax = axes_flat[i]
    ax.hist(pdf_n[col].dropna(), bins=40, color="#4C8BF5",
            alpha=0.6, density=True, label="Normal")
    ax.hist(pdf_a[col].dropna(), bins=40, color="#E8453C",
            alpha=0.7, density=True, label="Ataque")
    corr_val = corr_nuevas[corr_nuevas["feature"] == col]["corr_abs"].values[0]
    ax.set_title(f"{col}\n(corr={corr_val:.3f})",
                 fontsize=10, fontweight="bold")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=8)

for j in range(len(nuevas_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Distribución de nuevas features — Normal vs Ataque",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6 — Guardar Delta features_fisicos

In [0]:
# Convertir de vuelta a Spark para guardar en Delta
# Guardamos todas las features: originales normalizadas + nuevas
# timestamp_dt se conserva por si se necesita para joins futuros
# label se conserva como en los de red

feature_cols_final = SIGNAL_COLS + nuevas_cols

print(f"Features finales    : {len(feature_cols_final)}")
print(f"  Originales (norm) : {SIGNAL_COLS}")
print(f"  Flags binarios    : {flag_cols}")
print(f"  Ratios            : {ratio_cols}")

df_spark = spark.createDataFrame(
    pdf[["timestamp_dt", "label"] + feature_cols_final]
)

df_spark.repartition(32) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_FE_FISICOS)

print(f"\nGuardado en: {DELTA_FE_FISICOS}")

# Verificacion rapida
df_check = spark.read.format("delta").load(DELTA_FE_FISICOS)
print(f"Filas  : {df_check.count():,}")
print(f"Cols   : {len(df_check.columns)}")
display(df_check.groupBy("label").count().orderBy("label"))

## 7 — Resumen Feature Engineering

In [0]:
print("=" * 60)
print("RESUMEN — FEATURE ENGINEERING DATOS FISICOS")
print("=" * 60)
print(f"Sensor eliminado        : FIT601 (correlacion ~0)")
print(f"Sensores originales     : {len(SIGNAL_COLS)}  {SIGNAL_COLS}")
print(f"  → normalizados [0,1] con MinMaxScaler")
print(f"Flags binarios creados  : {len(flag_cols)}")
for f in flag_cols:
    corr = corr_nuevas[corr_nuevas['feature']==f]['corr_abs'].values[0]
    print(f"  {f:<30} corr={corr:.4f}")
print(f"Ratios creados          : {len(ratio_cols)}")
for r in ratio_cols:
    corr = corr_nuevas[corr_nuevas['feature']==r]['corr_abs'].values[0]
    print(f"  {r:<30} corr={corr:.4f}")
print("-" * 60)
print(f"Total features finales  : {len(feature_cols_final)}")
print(f"Top feature             : {corr_nuevas.iloc[0]['feature']}  "
      f"({corr_nuevas.iloc[0]['corr_abs']:.4f})")
print(f"Delta guardado          : {DELTA_FE_FISICOS}")
print("=" * 60)